# Notebook 55 — Deriving the Projection Weight from Fit Sensitivity

Goal:

Instead of fitting a weight function w(x), derive it from the **sensitivity of the fit**:

b ≈ ∫ w(x) b_local(x) dx

Hypothesis:

w(x) ∝ |∂ log S(x) / ∂ b|

This connects the projection directly to the fitting process.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Core definitions

In [ ]:
x = np.linspace(1e-4, 1.0, 800)

def build_S_from_gamma(gamma_x):
    dx = x[1] - x[0]
    integral = np.cumsum(gamma_x) * dx
    return np.exp(-integral), integral

def stretched(x, a, b):
    return np.exp(-a * x**b)

def fit_stretched(S):
    popt, _ = curve_fit(stretched, x, S, p0=[1.0, 1.0], maxfev=20000)
    return popt

def b_local(x, gamma_x, integral):
    return x * gamma_x / np.clip(integral, 1e-12, None)

## Example structured Γ(x)

In [ ]:
def gamma_wave(x):
    return 2.0 * (1 + 0.2*np.sin(2*np.pi*x)) * x**(-0.1)

gamma_x = gamma_wave(x)
S, I = build_S_from_gamma(gamma_x)
a_fit, b_fit = fit_stretched(S)
b_loc = b_local(x, gamma_x, I)

print("Fitted b:", b_fit)

## Sensitivity-derived weight

In [ ]:
# derivative of log S wrt b (numerical)
eps = 1e-4
S_plus = stretched(x, a_fit, b_fit + eps)
S_minus = stretched(x, a_fit, b_fit - eps)

dlogS_db = (np.log(S_plus) - np.log(S_minus)) / (2*eps)

w = np.abs(dlogS_db)
w = w / np.trapz(w, x)

## Projection vs fitted b

In [ ]:
b_proj = np.trapz(w * b_loc, x)
print("Projected b:", b_proj)

## Plot: local exponent + weight

In [ ]:
plt.figure()
plt.plot(x, b_loc, label="b_local(x)")
plt.plot(x, w / np.max(w), '--', label="weight (scaled)")
plt.legend()
plt.title("Local exponent field and derived weight")
plt.xlabel("x")
plt.ylabel("value")
plt.show()

## Plot: S(x) and sensitivity

In [ ]:
plt.figure()
plt.plot(x, S, label="S(x)")
plt.plot(x, np.abs(dlogS_db)/np.max(np.abs(dlogS_db)), '--', label="|∂logS/∂b| (scaled)")
plt.legend()
plt.title("Signal and sensitivity")
plt.xlabel("x")
plt.show()

## Conclusion

We derive:

w(x) ∝ |∂ log S(x) / ∂ b|

Thus:

b ≈ ∫ w(x) b_local(x) dx

The projection weight is not arbitrary — it is determined by the **fit sensitivity**.
